In [2]:
# ════════════════════════════════════════════════════════════════
# VÉRIFICATION COMPATIBILITÉ ESP32-S3 + CONVERSION .cc
# ════════════════════════════════════════════════════════════════
import tensorflow as tf
import numpy as np
import os
from pathlib import Path

OUTPUT_DIR = Path("C:/Users/asalou/S7/stages7/project/augmentation/aug")
# ── 1. Vérifier les opérations du modèle INT8 ─────────────────
print("1. VÉRIFICATION DES OPÉRATIONS TFLITE MICRO")
print("="*60)

# Opérations supportées par TFLite Micro sur ESP32-S3
OPS_SUPPORTEES = {
    'CONV_2D':                  '✅ Supporté',
    'DEPTHWISE_CONV_2D':        '✅ Supporté',
    'MAX_POOL_2D':              '✅ Supporté',
    'AVERAGE_POOL_2D':          '✅ Supporté',
    'FULLY_CONNECTED':          '✅ Supporté',
    'SOFTMAX':                  '✅ Supporté',
    'RESHAPE':                  '✅ Supporté',
    'ADD':                      '✅ Supporté',
    'MUL':                      '✅ Supporté',
    'MEAN':                     '✅ Supporté',
    'PAD':                      '✅ Supporté',
    'BATCH_NORM':               '✅ Fusionné dans Conv à l\'inférence',
    'DROPOUT':                  '✅ Supprimé à l\'inférence',
    'DEQUANTIZE':               '✅ Supporté',
    'QUANTIZE':                 '✅ Supporté',
}

# Charger le modèle INT8 et lister ses opérations
interpreter = tf.lite.Interpreter(
    model_path=f'{OUTPUT_DIR}/cnn2d_int8.tflite')
interpreter.allocate_tensors()

ops_utilisees = set()
for op in interpreter._get_ops_details():
    ops_utilisees.add(op['op_name'])

print(f"\nOpérations utilisées dans votre modèle :")
toutes_ok = True
for op in sorted(ops_utilisees):
    statut = OPS_SUPPORTEES.get(op, '⚠️  À vérifier')
    print(f"  {op:35s} : {statut}")
    if '⚠️' in statut:
        toutes_ok = False

print(f"\n{'✅ Toutes les opérations sont compatibles ESP32-S3' if toutes_ok else '⚠️  Certaines opérations nécessitent vérification'}")

1. VÉRIFICATION DES OPÉRATIONS TFLITE MICRO

Opérations utilisées dans votre modèle :
  ADD                                 : ✅ Supporté
  CONV_2D                             : ✅ Supporté
  DELEGATE                            : ⚠️  À vérifier
  FULLY_CONNECTED                     : ✅ Supporté
  MAX_POOL_2D                         : ✅ Supporté
  MEAN                                : ✅ Supporté
  MUL                                 : ✅ Supporté
  SOFTMAX                             : ✅ Supporté

⚠️  Certaines opérations nécessitent vérification


C:\Users\asalou\S7\envs\stages7\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [3]:
# ── 2. Estimer la RAM nécessaire ──────────────────────────────
print("\n2. ESTIMATION MÉMOIRE ESP32-S3")
print("="*60)

# Analyser les tenseurs du modèle
tensors     = interpreter.get_tensor_details()
ram_tensors = sum(
    np.prod(t['shape']) * np.dtype(t['dtype']).itemsize
    for t in tensors if len(t['shape']) > 0
)

# Taille du modèle
model_size = os.path.getsize(f'{OUTPUT_DIR}/cnn2d_int8.tflite')

# Activation la plus grande (buffer intermédiaire)
max_activation = max(
    np.prod(t['shape']) * np.dtype(t['dtype']).itemsize
    for t in tensors if len(t['shape']) > 0
)

print(f"\n  Taille modèle INT8      : {model_size/1024:>8.1f} KB  → Flash 8MB  ✅")
print(f"  RAM tenseurs total      : {ram_tensors/1024:>8.1f} KB")
print(f"  Buffer activation max   : {max_activation/1024:>8.1f} KB")
print(f"  RAM estimée totale      : {(ram_tensors + 50*1024)/1024:>8.1f} KB  (+ 50KB overhead)")
print(f"\n  SRAM interne ESP32-S3   :      512 KB")
print(f"  PSRAM externe           :    8 192 KB")

ram_totale = ram_tensors + 50*1024
if ram_totale < 512*1024:
    print(f"\n  ✅ Tient dans la SRAM interne ({ram_totale/1024:.0f} KB < 512 KB)")
elif ram_totale < 8192*1024:
    print(f"\n  ⚠️  Nécessite la PSRAM ({ram_totale/1024:.0f} KB > 512 KB)")
    print(f"     → L'ATOMS3R a 8MB PSRAM ✅ compatible")
else:
    print(f"\n  ❌ Trop grand même pour la PSRAM")


2. ESTIMATION MÉMOIRE ESP32-S3

  Taille modèle INT8      :    183.8 KB  → Flash 8MB  ✅
  RAM tenseurs total      :   1192.7 KB
  Buffer activation max   :    102.0 KB
  RAM estimée totale      :   1242.7 KB  (+ 50KB overhead)

  SRAM interne ESP32-S3   :      512 KB
  PSRAM externe           :    8 192 KB

  ⚠️  Nécessite la PSRAM (1243 KB > 512 KB)
     → L'ATOMS3R a 8MB PSRAM ✅ compatible


In [4]:
# ── 3. Convertir en tableau C (.cc) pour Arduino/ESP-IDF ──────
print("\n3. CONVERSION EN FICHIER .cc POUR ESP32-S3")
print("="*60)

# Lire le modèle INT8
with open(f'{OUTPUT_DIR}/cnn2d_int8.tflite', 'rb') as f:
    model_bytes = f.read()

# Convertir en tableau C hexadécimal
def bytes_to_c_array(data, var_name='cnn_model'):
    """Convertit bytes → tableau C pour TFLite Micro."""
    lines   = [f'// CNN 2D Miction — Modèle INT8 pour ESP32-S3']
    lines  += [f'// Taille : {len(data)} bytes ({len(data)/1024:.1f} KB)']
    lines  += [f'// Généré automatiquement — ne pas modifier']
    lines  += [f'']
    lines  += [f'#include <stdint.h>']
    lines  += [f'']
    lines  += [f'const unsigned char {var_name}[] __attribute__((aligned(8))) = {{']

    # 12 bytes par ligne
    for i in range(0, len(data), 12):
        chunk = data[i:i+12]
        hex_vals = ', '.join(f'0x{b:02x}' for b in chunk)
        lines.append(f'  {hex_vals},')

    lines += ['};']
    lines += [f'']
    lines += [f'const int {var_name}_len = {len(data)};']

    return '\n'.join(lines)

# Générer le .cc
c_code    = bytes_to_c_array(model_bytes, 'cnn_miction_model')
path_cc   = f'{OUTPUT_DIR}/cnn_miction_model.cc'
with open(path_cc, 'w') as f:
    f.write(c_code)

# Générer le header .h
h_code = f"""// CNN 2D Miction — Header pour ESP32-S3
#ifndef CNN_MICTION_MODEL_H
#define CNN_MICTION_MODEL_H

#include <stdint.h>

extern const unsigned char cnn_miction_model[];
extern const int cnn_miction_model_len;

// Paramètres du modèle
#define MODEL_INPUT_H     64    // Hauteur spectrogramme Mel
#define MODEL_INPUT_W     51    // Largeur (frames temporelles)
#define MODEL_INPUT_C      1    // Canaux (mono)
#define MODEL_N_CLASSES    4    // Nombre de classes
#define MODEL_SEGMENT_MS 500    // Durée segment en ms
#define MODEL_HOP_MS     250    // Hop chevauchement en ms
#define MODEL_SR       16000    // Fréquence échantillonnage
#define MODEL_N_MELS      64    // Bandes Mel
#define MODEL_N_FFT      512    // Taille FFT
#define MODEL_HOP_LEN    160    // Hop longueur Mel

// Classes (ordre LabelEncoder)
// 0 = bruit_ambiant
// 1 = bruit_chasse
// 2 = miction_active
// 3 = silence

#endif // CNN_MICTION_MODEL_H
"""

path_h = f'{OUTPUT_DIR}/cnn_miction_model.h'
with open(path_h, 'w') as f:
    f.write(h_code)

print(f"\n  ✅ Fichiers générés :")
print(f"     {path_cc}  ({os.path.getsize(path_cc)/1024:.0f} KB)")
print(f"     {path_h}   ({os.path.getsize(path_h)/1024:.0f} KB)")


3. CONVERSION EN FICHIER .cc POUR ESP32-S3

  ✅ Fichiers générés :
     C:\Users\asalou\S7\stages7\project\augmentation\aug/cnn_miction_model.cc  (1149 KB)
     C:\Users\asalou\S7\stages7\project\augmentation\aug/cnn_miction_model.h   (1 KB)

  Ces fichiers s'intègrent dans votre projet
  Arduino IDE ou ESP-IDF pour l'ATOMS3R.


In [1]:
# ── 4. Résumé déploiement ATOMS3R ─────────────────────────────
print("\n4. RÉSUMÉ DÉPLOIEMENT ATOMS3R")
print("="*60)
print(f"""
  Hardware :
    Module   : M5Stack ATOMS3R
    CPU      : ESP32-S3 @ 240MHz dual-core
    Flash    : 8 MB  → modèle INT8 (184 KB) 
    PSRAM    : 8 MB  → tenseurs + buffers   
    SRAM     : 512 KB

  Pipeline embarqué :
    1. Acquisition audio   (microphone I2S)
    2. Filtrage Butterworth (100-6000 Hz)
    3. Segmentation 500ms  (hop 250ms)
    4. Spectrogramme Mel   (64×51)
    5. Inférence CNN INT8  (~2-5ms/segment)
    6. Correction contexte (fenêtre=1)
    7. Indicateurs médicaux (durée + type)
    8. Affichage écran 0.85\"

  Fichiers à intégrer dans le projet Arduino :
    cnn_miction_model.cc   ← modèle INT8
    cnn_miction_model.h    ← constantes
    Librairie : tflite-micro (ESP32)
""")


4. RÉSUMÉ DÉPLOIEMENT ATOMS3R

  Hardware :
    Module   : M5Stack ATOMS3R
    CPU      : ESP32-S3 @ 240MHz dual-core
    Flash    : 8 MB  → modèle INT8 (184 KB) 
    PSRAM    : 8 MB  → tenseurs + buffers   
    SRAM     : 512 KB

  Pipeline embarqué :
    1. Acquisition audio   (microphone I2S)
    2. Filtrage Butterworth (100-6000 Hz)
    3. Segmentation 500ms  (hop 250ms)
    4. Spectrogramme Mel   (64×51)
    5. Inférence CNN INT8  (~2-5ms/segment)
    6. Correction contexte (fenêtre=1)
    7. Indicateurs médicaux (durée + type)
    8. Affichage écran 0.85"

  Fichiers à intégrer dans le projet Arduino :
    cnn_miction_model.cc   ← modèle INT8
    cnn_miction_model.h    ← constantes
    Librairie : tflite-micro (ESP32)

